# Data exploration — IMDb

Owner: Mathis. Les huit questions sont traitées avec **PySpark**, sur la couche
**silver** produite par `01_data_preparation.ipynb` et les helpers de `jobs/common.py`.

**Exécution :** ouvrir ce fichier dans le Jupyter du projet, après la préparation,
puis **Restart Kernel and Run All Cells**. Les résultats complets sont exportés dans
`results/exploration/<horodatage>/`, en Parquet et CSV distribués (plusieurs fichiers
`part-*`). Les tableaux affichés sont seulement des aperçus. Aucun résultat IMDb
n'est prérempli : les tests sur données synthétiques ne sont pas des résultats réels.

Les choix opérationnels ci-dessous sont des paramètres à discuter dans le groupe.
Ils ne constituent pas le texte du rapport à remettre : `REPORT.md` doit être rédigé
par les étudiants, conformément à `CONTRIBUTING.md`. Code préparé avec Codex (OpenAI).

In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timezone
import json

# Fonctionne depuis la racine du dépôt ou depuis notebooks/.
PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / "jobs" / "common.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Placer ce notebook dans le dossier notebooks/ du projet IMDb.")
sys.path.insert(0, str(PROJECT / "jobs"))

from pyspark.sql import functions as F, Window
from pyspark.sql.types import ArrayType, MapType, StructType
from common import get_spark, read_silver, SILVER_DIR

spark = get_spark("imdb_data_exploration", driver_memory="2g", cores=2)
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.shuffle.partitions", "200")

## Paramètres et conventions

- « Movie » utilise `is_movie` ; « feature-length » utilise `is_feature_film` (>40 min).
- Les titres signalés `is_unreleased` sont exclus ; aucune note manquante n'est imputée.
- Q1 : écart absolu d'au moins deux écarts-types ; au moins 30 films par genre.
  Les durées suspectes restent visibles, avec leur indicateur de qualité.
- Q2–Q3 : popularité = somme des votes actuels des films de chaque année de sortie.
  Ce n'est **pas** la popularité historique mesurée cette année-là. Le classement
  par nombre de films est aussi exporté pour distinguer popularité et production.
- Q4 : moyenne arithmétique des notes des autres films notés ; au moins deux autres films.
- Q5 : au moins trois saisons régulières, **chaque saison observée** ayant ≥10 épisodes.
  Saison 0 exclue. Les ex æquo sont conservés ; les extrema concernent les épisodes notés.
- Q6 : succès = note ≥7 et ≥1 000 votes. La langue du **titre original** sert de proxy
  incomplet, jamais de preuve de la langue parlée ; aucun repli sur les titres traduits.
- Q7 : début = premier quart de la carrière observée ; au moins trois films notés
  de chaque côté, avec ≥100 votes, pour comparer les périodes.
- Q8 : rôles étudiés = acteur/actrice (regroupés), réalisateur et scénariste.

Les carrières sont limitées aux crédits principaux disponibles dans IMDb.

In [ ]:
PARAMETERS = dict(
    start_year=1980, end_year=2025, runtime_z=2.0, min_genre_movies=30,
    success_rating=7.0, success_votes=1000,
    career_years=20, career_movies=15, early_fraction=0.25,
    career_votes=100, min_period_rated=3,
)

In [ ]:
TABLES = ["title.basics", "title.ratings", "title.crew", "title.episode",
          "title.principals", "title.akas", "name.basics"]
missing = [name for name in TABLES if not (SILVER_DIR / name).is_dir()]
if missing:
    raise FileNotFoundError(
        f"Tables silver absentes dans {SILVER_DIR}: {missing}. "
        "Exécuter d'abord le téléchargement, la conversion bronze et le notebook 01.")
silver = {name: read_silver(spark, name) for name in TABLES}
titles = silver["title.basics"].select(
    "tconst", "primaryTitle", "titleType", "startYear", "runtimeMinutes", "genres",
    "is_movie", "is_feature_film", "is_unreleased", "is_runtime_suspicious")
ratings = silver["title.ratings"].select("tconst", "averageRating", "numVotes")
names = silver["name.basics"].select("nconst", "primaryName")
crew = silver["title.crew"].select("tconst", "directors", "writers")
principals = silver["title.principals"].select("tconst", "nconst", "category")
episodes = silver["title.episode"]
akas = silver["title.akas"]
movies = titles.filter(F.col("is_movie") & ~F.col("is_unreleased"))
movie_ratings = movies.join(ratings, "tconst", "left")
cast = (principals.filter(F.col("category").isin("actor", "actress"))
        .select("tconst", "nconst").dropDuplicates()
        .join(movies.select("tconst"), "tconst", "left_semi"))

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RESULTS_DIR = PROJECT / "results" / "exploration" / RUN_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=False)
(RESULTS_DIR / "parameters.json").write_text(
    json.dumps({**PARAMETERS, "silver_dir": str(SILVER_DIR), "run_utc": RUN_ID}, indent=2),
    encoding="utf-8")
print(f"Exports : {RESULTS_DIR}")

def save_result(df, name, order_by, preview=20):
    """Matérialise une fois ; export complet distribué, aperçu borné et ordonné."""
    target = RESULTS_DIR / name
    df.write.mode("errorifexists").parquet(str(target / "parquet"))
    stored = spark.read.parquet(str(target / "parquet"))
    csv_ready = stored.select(*[
        F.to_json(F.col(field.name)).alias(field.name)
        if isinstance(field.dataType, (ArrayType, MapType, StructType))
        else F.col(field.name) for field in stored.schema.fields
    ])
    csv_ready.write.mode("errorifexists").option("header", True).csv(str(target / "csv"))
    print(f"{name}: {stored.count():,} lignes (export complet)")
    stored.orderBy(*order_by).show(preview, truncate=False)
    return stored

movie_ratings.agg(
    F.count("*").alias("movies"), F.count("averageRating").alias("rated_movies"),
    F.sum(F.col("startYear").isNull().cast("long")).alias("missing_year"),
    F.sum(F.col("genres").isNull().cast("long")).alias("missing_genres"),
    F.sum(F.col("runtimeMinutes").isNull().cast("long")).alias("missing_runtime"),
).show(truncate=False)

## 1. Find all feature-length movies by genre whose length is notably different than the average length of movies in that genre.

Une ligne par couple film–genre. La référence est l'ensemble des longs métrages
du genre, sans restriction d'année. `z_runtime` indique le sens et l'amplitude de
l'écart. Un genre à variance nulle ou à effectif insuffisant est exclu.
Le seuil est descriptif : il ne constitue pas un test statistique de normalité.

In [ ]:
def unusual_runtimes(movies, z_threshold=2.0, min_count=30):
    feature_genres = (movies.filter(F.col("is_feature_film"))
        .withColumn("genre", F.explode(F.array_distinct("genres")))
        .filter(F.col("genre").isNotNull() & (F.trim("genre") != "")))
    stats = feature_genres.groupBy("genre").agg(
        F.count("*").alias("genre_movies"),
        F.avg("runtimeMinutes").alias("genre_mean_minutes"),
        F.stddev_pop("runtimeMinutes").alias("genre_std_minutes"))
    return (feature_genres.join(stats, "genre")
        .filter((F.col("genre_movies") >= min_count) & (F.col("genre_std_minutes") > 0))
        .withColumn("difference_minutes", F.col("runtimeMinutes") - F.col("genre_mean_minutes"))
        .withColumn("z_runtime", F.col("difference_minutes") / F.col("genre_std_minutes"))
        .filter(F.abs("z_runtime") >= z_threshold)
        .select("genre", "tconst", "primaryTitle", "startYear", "runtimeMinutes",
                "genre_movies", "genre_mean_minutes", "genre_std_minutes",
                "difference_minutes", "z_runtime", "is_runtime_suspicious"))

In [ ]:
q1 = save_result(unusual_runtimes(movies, PARAMETERS["runtime_z"], PARAMETERS["min_genre_movies"]),
                 "q1_unusual_runtimes", [F.desc(F.abs("z_runtime")), "genre", "tconst"])

## 2. What are the top three most popular movie genres for each year between 1980 and 2025?

Chaque film compte dans chacun de ses genres, une seule fois par genre. Les films
sans note comptent dans la production, mais apportent zéro vote observé.
Les ex æquo sont départagés par ordre alphabétique pour obtenir exactement trois
genres (si disponibles) par année ; aucun seuil de votes n'est imposé ici.

In [ ]:
def genre_rankings(movie_ratings, start_year=1980, end_year=2025):
    film_genres = (movie_ratings.filter(F.col("startYear").between(start_year, end_year))
        .withColumn("genre", F.explode(F.array_distinct("genres")))
        .filter(F.col("genre").isNotNull() & (F.trim("genre") != "")))
    stats = film_genres.groupBy("startYear", "genre").agg(
        F.count("*").alias("movie_count"),
        F.count("averageRating").alias("rated_movie_count"),
        F.sum(F.coalesce(F.col("numVotes"), F.lit(0)).cast("long")).alias("total_votes"),
        F.avg("averageRating").alias("mean_rating"),
        F.sum(F.col("averageRating") * F.col("numVotes")).alias("rating_vote_sum"))
    return (stats
        .withColumn("vote_weighted_rating", F.when(F.col("total_votes") > 0,
                    F.col("rating_vote_sum") / F.col("total_votes")))
        .drop("rating_vote_sum")
        .withColumn("popularity_rank", F.row_number().over(
            Window.partitionBy("startYear").orderBy(F.desc("total_votes"), "genre")))
        .withColumn("production_rank", F.row_number().over(
            Window.partitionBy("startYear").orderBy(F.desc("movie_count"), "genre"))))

In [ ]:
genre_stats = save_result(genre_rankings(movie_ratings, PARAMETERS["start_year"], PARAMETERS["end_year"]),
                          "q2_all_genres", ["startYear", "popularity_rank"])
q2 = save_result(genre_stats.filter(F.col("popularity_rank") <= 3),
                 "q2_top_three_by_votes", ["startYear", "popularity_rank"], preview=138)
save_result(genre_stats.filter(F.col("production_rank") <= 3),
            "q2_top_three_by_movie_count", ["startYear", "production_rank"])
years_present = genre_stats.select("startYear").distinct()
spark.range(PARAMETERS["start_year"], PARAMETERS["end_year"] + 1).select(
    F.col("id").alias("startYear")).join(years_present, "startYear", "left_anti").show()

## 3. Compare the results from the previous question. How do the top three genres compare to the rest of the genres of movies in a given year?

Deux lectures complémentaires :

1. **Genres** : poids du top 3 dans les sommes par genre, note moyenne par genre
   et votes moyens par film. Ces parts portent sur des appartenances film–genre :
   elles ne représentent pas des parts de films uniques.
2. **Films uniques** : un film appartient au groupe top 3 dès qu'il possède au moins
   un de ces genres ; sinon au reste. Les films sans genre forment un troisième groupe.
   Les groupes sont disjoints, donc les films et votes ne sont pas comptés deux fois.

In [ ]:
def compare_genres(genre_stats, movie_ratings, start_year=1980, end_year=2025):
    by_genre = (genre_stats.withColumn("group", F.when(F.col("popularity_rank") <= 3, "top3")
                                    .otherwise("other_genres"))
        .groupBy("startYear", "group").agg(
            F.count("*").alias("genre_count"),
            F.sum("movie_count").alias("film_genre_memberships"),
            F.sum("total_votes").alias("genre_vote_sum"),
            F.avg("mean_rating").alias("unweighted_mean_of_genre_ratings"))
        .withColumn("membership_vote_share_pct", 100 * F.col("genre_vote_sum") /
            F.when(F.sum("genre_vote_sum").over(Window.partitionBy("startYear")) > 0,
                   F.sum("genre_vote_sum").over(Window.partitionBy("startYear"))))
        .withColumn("votes_per_membership", F.col("genre_vote_sum") / F.col("film_genre_memberships")))
    top = (genre_stats.filter(F.col("popularity_rank") <= 3).groupBy("startYear")
           .agg(F.collect_set("genre").alias("top_genres")))
    grouped = (movie_ratings.filter(F.col("startYear").between(start_year, end_year))
        .join(top, "startYear", "left")
        .withColumn("group", F.when(F.col("genres").isNull() | (F.size("genres") == 0), "unknown_genre")
            .when(F.size(F.array_intersect("genres", "top_genres")) > 0, "has_top3_genre")
            .otherwise("other_genres_only")))
    by_movie = grouped.groupBy("startYear", "group").agg(
        F.count("*").alias("movie_count"), F.count("averageRating").alias("rated_movie_count"),
        F.sum(F.coalesce(F.col("numVotes"), F.lit(0)).cast("long")).alias("total_votes"),
        F.avg("averageRating").alias("mean_rating"))
    return by_genre, (by_movie
        .withColumn("movie_share_pct", 100 * F.col("movie_count") /
                    F.sum("movie_count").over(Window.partitionBy("startYear")))
        .withColumn("vote_share_pct", F.when(
            F.sum("total_votes").over(Window.partitionBy("startYear")) > 0,
            100 * F.col("total_votes") / F.sum("total_votes").over(Window.partitionBy("startYear"))))
        .withColumn("votes_per_movie", F.col("total_votes") / F.col("movie_count")))

In [ ]:
q3_genres, q3_movies = compare_genres(genre_stats, movie_ratings,
                                    PARAMETERS["start_year"], PARAMETERS["end_year"])
save_result(q3_genres, "q3_genre_comparison", ["startYear", "group"])
save_result(q3_movies, "q3_disjoint_movie_comparison", ["startYear", "group"])

## 4. List all directors who have directed exactly one movie with a rating >= 8.0 but who have multiple other movies directed with an average rating of <= 6.0.

Les réalisateurs viennent de `title.crew`, pas uniquement des principaux crédits.
« Exactement un » porte sur les films notés observables ; les films sans note sont
comptés séparément. Aucune note minimale par nombre de votes n'est ajoutée à l'énoncé.

In [ ]:
def exceptional_directors(crew, movie_ratings, names):
    directed = (crew.select("tconst", F.explode("directors").alias("nconst"))
                .dropDuplicates().join(movie_ratings, "tconst"))
    high = F.col("averageRating") >= 8.0
    stats = directed.groupBy("nconst").agg(
        F.count("*").alias("all_movies"), F.count("averageRating").alias("rated_movies"),
        F.sum(F.when(high, 1).otherwise(0)).alias("high_rated_movies"),
        F.sum(F.when(F.col("averageRating") < 8.0, 1).otherwise(0)).alias("other_rated_movies"),
        F.avg(F.when(F.col("averageRating") < 8.0, F.col("averageRating"))).alias("other_mean_rating"))
    qualified = (stats.filter((F.col("high_rated_movies") == 1) &
                 (F.col("other_rated_movies") >= 2) & (F.col("other_mean_rating") <= 6.0))
                 .withColumn("unrated_movies", F.col("all_movies") - F.col("rated_movies")))
    hit = directed.filter(high).select("nconst", F.col("tconst").alias("hit_tconst"),
        F.col("primaryTitle").alias("hit_title"), F.col("averageRating").alias("hit_rating"),
        F.col("numVotes").alias("hit_votes"))
    return qualified.join(names, "nconst", "left").join(hit, "nconst")

In [ ]:
q4 = save_result(exceptional_directors(crew, movie_ratings, names),
                 "q4_directors", ["other_mean_rating", "nconst"])

## 5. Identify television series with at least three seasons consisting of >=10 episodes where the worst episode occurred in the last season and the best episode occurred within the first two seasons.

Le nombre de saisons et la dernière saison sont déterminés **avant** la jointure
aux notes, pour ne pas faire disparaître une saison peu notée. « Dernière » signifie
la dernière saison présente dans le jeu, éventuellement en cours. Les séries avec
un numéro de saison inconnu sont écartées (leur dernière saison serait ambiguë).
Les minima/maxima peuvent aussi être atteints ailleurs : on exige au moins une
occurrence aux positions demandées et on exporte toutes les occurrences conformes.

In [ ]:
def qualifying_series(titles, episodes, ratings):
    series = titles.filter(F.col("titleType").isin("tvSeries", "tvMiniSeries") &
                           ~F.col("is_unreleased")).select(
        F.col("tconst").alias("parentTconst"), F.col("primaryTitle").alias("series_title"))
    raw = episodes.join(series, "parentTconst").dropDuplicates(["tconst"])
    ambiguous = raw.filter(F.col("seasonNumber").isNull() | (F.col("seasonNumber") < 0))                    .select("parentTconst").distinct()
    regular = raw.filter(F.col("seasonNumber") > 0)
    seasons = regular.groupBy("parentTconst", "seasonNumber").agg(F.count("*").alias("episodes_in_season"))
    eligible = (seasons.groupBy("parentTconst").agg(
        F.count("*").alias("season_count"), F.min("episodes_in_season").alias("min_season_episodes"),
        F.max("seasonNumber").alias("last_season"), F.sum("episodes_in_season").alias("episode_count"))
        .filter((F.col("season_count") >= 3) & (F.col("min_season_episodes") >= 10))
        .join(ambiguous, "parentTconst", "left_anti"))
    rated = regular.join(ratings, "tconst").join(eligible, "parentTconst")
    extrema = rated.groupBy("parentTconst").agg(
        F.min("averageRating").alias("worst_rating"), F.max("averageRating").alias("best_rating"),
        F.count("*").alias("rated_episode_count"))
    marked = rated.join(extrema, "parentTconst")
    best = marked.filter((F.col("averageRating") == F.col("best_rating")) &
                          F.col("seasonNumber").isin(1, 2)).select(
        "parentTconst", "tconst", "seasonNumber", "episodeNumber", "averageRating", "numVotes")
    worst = marked.filter((F.col("averageRating") == F.col("worst_rating")) &
                           (F.col("seasonNumber") == F.col("last_season"))).select(best.columns)
    selected = best.select("parentTconst").distinct().join(worst.select("parentTconst").distinct(), "parentTconst")
    result = (eligible.join(selected, "parentTconst").join(series, "parentTconst")
        .join(extrema, "parentTconst")
        .withColumn("rated_episode_pct", 100 * F.col("rated_episode_count") / F.col("episode_count")))
    evidence = (best.withColumn("extreme", F.lit("best_in_first_two"))
        .unionByName(worst.withColumn("extreme", F.lit("worst_in_last")))
        .join(selected, "parentTconst"))
    return result, evidence

In [ ]:
q5, q5_evidence = qualifying_series(titles, episodes, ratings)
save_result(q5, "q5_series", ["parentTconst"])
save_result(q5_evidence, "q5_episode_evidence", ["parentTconst", "extreme", "tconst"])

## 6. Find any actors/actresses who have achieved success across films in multiple distinct languages (e.g. been in a successful film originally in French and English). Additionally, list all actors/actresses who have achieved success in movies in >=3 languages (e.g. French, German, and English).

**Limite de la source :** [`title.akas.language`](https://developer.imdb.com/non-commercial-datasets/)
décrit la langue d'un titre localisé. Un film américain traduit en français ne prouve
pas que ses acteurs ont joué dans un film francophone. On ne peut donc pas répondre
définitivement à la question des langues originales parlées avec ces seules tables.

Le calcul ci-dessous fournit une **liste candidate**, fondée uniquement sur les
entrées `isOriginalTitle = 1` avec une langue renseignée (hors codes indéterminés).
Il exporte les preuves film–langue, les candidats à ≥2 et à ≥3 langues et la couverture.
Une validation par une source de langues originales est nécessaire avant toute conclusion.
Si aucune entrée originale n'a de langue exploitable, une sortie vide est correcte.

In [ ]:
def multilingual_candidates(cast, movie_ratings, akas, names, min_rating=7.0, min_votes=1000):
    successful = movie_ratings.filter((F.col("averageRating") >= min_rating) & (F.col("numVotes") >= min_votes))
    languages = (akas.filter(F.col("isOriginalTitle") == 1)
        .select(F.col("titleId").alias("tconst"), F.lower(F.trim("language")).alias("language"))
        .filter(F.col("language").isNotNull() & ~F.col("language").isin("", "und", "mul", "zxx", "\\n"))
        .dropDuplicates())
    evidence = (cast.join(successful.select("tconst", "primaryTitle", "averageRating", "numVotes"), "tconst")
        .join(languages, "tconst").dropDuplicates(["nconst", "tconst", "language"]))
    summary = (evidence.groupBy("nconst").agg(
        F.countDistinct("language").alias("language_count"),
        F.sort_array(F.collect_set("language")).alias("original_title_languages"),
        F.countDistinct("tconst").alias("successful_movie_count"))
        .filter(F.col("language_count") >= 2).join(names, "nconst", "left")
        .withColumn("evidence_status", F.lit("original-title-language proxy; spoken language unverified")))
    coverage = (successful.select("tconst").join(languages.select("tconst").distinct()
                .withColumn("has_language", F.lit(1)), "tconst", "left")
        .agg(F.count("*").alias("successful_movies"), F.count("has_language").alias("with_original_title_language"))
        .withColumn("coverage_pct", F.when(F.col("successful_movies") > 0,
            100 * F.col("with_original_title_language") / F.col("successful_movies"))))
    return summary, evidence.join(summary.select("nconst"), "nconst"), coverage

In [ ]:
q6, q6_evidence, q6_coverage = multilingual_candidates(cast, movie_ratings, akas, names,
    PARAMETERS["success_rating"], PARAMETERS["success_votes"])
save_result(q6_coverage, "q6_language_coverage", ["successful_movies"])
save_result(q6, "q6_candidates_two_or_more_languages", [F.desc("language_count"), "nconst"])
save_result(q6.filter(F.col("language_count") >= 3), "q6_candidates_three_or_more_languages", ["nconst"])
save_result(q6_evidence, "q6_language_evidence", ["nconst", "language", "tconst"])

## 7. Find all actors/actresses who have been in movies for 20+ years and at least 15 or more movies whose popularity peaked early in their career (e.g. had their highest rated movies early in their career as compared to later.)

La durée de carrière est `dernière année − première année` sur tous les films datés
où la personne est acteur/actrice, y compris non notés. Le minimum de 15 films porte
sur ces films datés distincts. Les films non datés sont comptés à part.

Le pic précoce exige une meilleure note strictement supérieure dans le premier quart
de carrière **et** une moyenne des notes supérieure à celle de la suite. Un maximum
atteint aussi tardivement ne passe pas ce critère. « Popularité » est ici approchée
par les notes, comme dans l'exemple de l'énoncé, et non par le volume de votes.

In [ ]:
def early_peak_actors(cast, movie_ratings, names, min_years=20, min_movies=15,
                     early_fraction=0.25, min_votes=100, min_period_rated=3):
    filmography = cast.join(movie_ratings, "tconst")
    careers = filmography.groupBy("nconst").agg(
        F.min("startYear").alias("first_year"), F.max("startYear").alias("last_year"),
        F.count("startYear").alias("dated_movies"),
        F.sum(F.col("startYear").isNull().cast("long")).alias("undated_movies"))
    careers = (careers.withColumn("career_years", F.col("last_year") - F.col("first_year"))
        .filter((F.col("career_years") >= min_years) & (F.col("dated_movies") >= min_movies))
        .withColumn("early_end_year", F.col("first_year") + F.floor(F.col("career_years") * early_fraction)))
    observed = (filmography.join(careers, "nconst")
        .filter(F.col("startYear").isNotNull() & F.col("averageRating").isNotNull() & (F.col("numVotes") >= min_votes))
        .withColumn("early", F.col("startYear") <= F.col("early_end_year")))
    stats = observed.groupBy("nconst").agg(*[
        expr for label, condition in [("early", F.col("early")), ("later", ~F.col("early"))]
        for expr in [F.count(F.when(condition, 1)).alias(label + "_rated_movies"),
                     F.avg(F.when(condition, F.col("averageRating"))).alias(label + "_mean_rating"),
                     F.max(F.when(condition, F.col("averageRating"))).alias(label + "_max_rating")]])
    selected = (stats.filter((F.col("early_rated_movies") >= min_period_rated) &
        (F.col("later_rated_movies") >= min_period_rated) &
        (F.col("early_max_rating") > F.col("later_max_rating")) &
        (F.col("early_mean_rating") > F.col("later_mean_rating")))
        .withColumn("mean_rating_drop", F.col("early_mean_rating") - F.col("later_mean_rating")))
    peaks = (observed.join(selected.select("nconst", "early_max_rating"), "nconst")
        .filter(F.col("early") & (F.col("averageRating") == F.col("early_max_rating")))
        .select("nconst", "tconst", "primaryTitle", "startYear", "averageRating", "numVotes"))
    return selected.join(careers, "nconst").join(names, "nconst", "left"), peaks

In [ ]:
q7, q7_peaks = early_peak_actors(cast, movie_ratings, names,
    PARAMETERS["career_years"], PARAMETERS["career_movies"], PARAMETERS["early_fraction"],
    PARAMETERS["career_votes"], PARAMETERS["min_period_rated"])
save_result(q7, "q7_early_peak_actors", [F.desc("mean_rating_drop"), "nconst"])
save_result(q7_peaks, "q7_peak_movies", ["nconst", "startYear", "tconst"])

## 8. Identify film professionals who can be considered "multi-talented" (e.g. some combo of writer/director/actor). How do these individuals perform when performing multiple roles on the same project (e.g. directing and acting in the same movie)?

On utilise des crédits effectifs par film, pas `primaryProfession`. Les crédits de
réalisation/écriture de `title.crew` complètent `title.principals`. Les doublons sont
supprimés par personne–film–rôle. Une personne est polyvalente si elle a au moins
deux rôles distincts sur l'ensemble de sa filmographie observée.

On compare ensuite, pour chaque personne, les films avec **un rôle** à ceux avec
**plusieurs rôles simultanés** : effectifs, couverture des notes, note moyenne,
note pondérée par votes et nombre moyen de votes. L'écart individuel n'est calculé
que lorsque les deux groupes ont des films notés. Le bilan agrège ces écarts en
donnant le même poids à chaque personne comparable ; il ne prouve pas un effet causal.

In [ ]:
def multitalented_professionals(principals, crew, movie_ratings, names):
    principal_roles = (principals.filter(F.col("category").isin("actor", "actress", "director", "writer"))
        .select("tconst", "nconst", F.when(F.col("category").isin("actor", "actress"), "actor")
                .otherwise(F.col("category")).alias("role")))
    directors = crew.select("tconst", F.explode("directors").alias("nconst")).withColumn("role", F.lit("director"))
    writers = crew.select("tconst", F.explode("writers").alias("nconst")).withColumn("role", F.lit("writer"))
    roles = (principal_roles.unionByName(directors).unionByName(writers).dropDuplicates()
             .join(movie_ratings.select("tconst"), "tconst", "left_semi"))
    people = (roles.groupBy("nconst").agg(F.sort_array(F.collect_set("role")).alias("career_roles"))
        .filter(F.size("career_roles") >= 2).join(names, "nconst", "left"))
    projects = (roles.groupBy("nconst", "tconst").agg(F.sort_array(F.collect_set("role")).alias("project_roles"))
        .join(people.select("nconst"), "nconst")
        .withColumn("role_count", F.size("project_roles"))
        .withColumn("role_mode", F.when(F.col("role_count") >= 2, "multiple").otherwise("single"))
        .join(movie_ratings.select("tconst", "primaryTitle", "averageRating", "numVotes"), "tconst"))
    comparison = projects.groupBy("nconst", "role_mode").agg(
        F.count("*").alias("movies"), F.count("averageRating").alias("rated_movies"),
        F.avg("averageRating").alias("mean_rating"), F.avg("numVotes").alias("mean_votes_rated_movies"),
        F.sum("numVotes").alias("total_votes"),
        F.sum(F.col("averageRating") * F.col("numVotes")).alias("weighted_sum"))
    comparison = (comparison.withColumn("vote_weighted_rating", F.when(F.col("total_votes") > 0,
        F.col("weighted_sum") / F.col("total_votes"))).drop("weighted_sum")
        .withColumn("rated_pct", 100 * F.col("rated_movies") / F.col("movies"))
        .join(people, "nconst"))
    paired = comparison.groupBy("nconst").agg(
        F.max(F.when(F.col("role_mode") == "single", F.col("mean_rating"))).alias("single_mean_rating"),
        F.max(F.when(F.col("role_mode") == "multiple", F.col("mean_rating"))).alias("multiple_mean_rating"))
    paired = (paired.filter(F.col("single_mean_rating").isNotNull() & F.col("multiple_mean_rating").isNotNull())
        .withColumn("multiple_minus_single", F.col("multiple_mean_rating") - F.col("single_mean_rating"))
        .join(people, "nconst"))
    overall = paired.agg(F.count("*").alias("comparable_people"),
        F.avg("multiple_minus_single").alias("mean_person_level_rating_difference"),
        F.expr("percentile_approx(multiple_minus_single, 0.5)").alias("median_person_level_rating_difference"))
    return people, projects, comparison, paired, overall

In [ ]:
q8_people, q8_projects, q8_comparison, q8_paired, q8_overall = multitalented_professionals(
    principals, crew, movie_ratings, names)
save_result(q8_people, "q8_multitalented_people", ["nconst"])
save_result(q8_projects.filter(F.col("role_count") >= 2), "q8_multiple_role_movies", ["nconst", "tconst"])
save_result(q8_comparison, "q8_per_person_comparison", ["nconst", "role_mode"])
save_result(q8_paired, "q8_paired_differences", [F.desc("multiple_minus_single"), "nconst"])
save_result(q8_overall, "q8_overall_comparison", ["comparable_people"])

## Vérification avant remise

- Relancer toutes les cellules sur le même snapshot silver et conserver les sorties.
- Examiner les effectifs et la couverture des notes avant d'interpréter les classements.
- Q6 reste une liste candidate : ne pas présenter ses langues comme des langues parlées vérifiées.
- Rédiger vous-mêmes les observations et la justification des choix dans `REPORT.md`.
- Déclarer l'aide de Codex dans le tableau « Use of AI / LLM tools » de `PROJECT.md`.
- Intégrer sur une branche `feature/exploration-...`, puis faire relire la PR par Oscar.

Sources : énoncé des huit cellules du notebook initial, `CONTRIBUTING.md`, schémas et
conventions de `jobs/common.py`, et [documentation IMDb](https://developer.imdb.com/non-commercial-datasets/).